# some subjects have a mismatch in paradigm and brain data!

--> change ` drop_no_responses=False` and now works!
behavior = sub.get_behavior(sessions=None, drop_no_responses=False).reset_index('session') # session will be range

& subList for rerunning at bottom of script

Number of voxels: 33
Traceback (most recent call last):
  File "/home/mrenke/git/stress_risk/stress_risk/fmri_analysis/encoding_model/fit_regression_encoding_model.py", line 121, in <module>
    main(args.subject, model_label=args.model_label, smoothed=args.smoothed, bids_folder=args.bids_folder, debug=args.debug) # , gaussian=not args.log_space
  File "/home/mrenke/git/stress_risk/stress_risk/fmri_analysis/encoding_model/fit_regression_encoding_model.py", line 73, in main
    data = pd.DataFrame(data, index=paradigm.index)
  File "/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/pandas/core/frame.py", line 827, in __init__
    mgr = ndarray_to_mgr(
  File "/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/pandas/core/internals/construction.py", line 336, in ndarray_to_mgr
    _check_values_indices_shape_match(values, index, columns)
  File "/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/pandas/core/internals/construction.py", line 420, in _check_values_indices_shape_match
    raise ValueError(f"Shape of passed values is {passed}, indices imply {implied}")
ValueError: Shape of passed values is (240, 1093), indices imply (238, 1093)

In [1]:
import os
import os.path as op
import argparse
from stress_risk.utils.data import Subject
import numpy as np
from braincoder.utils import get_rsq
import pandas as pd
from models_sessionRegressor import get_paradigm, get_model, fit_model, get_conditionspecific_parameters
from nilearn.maskers import NiftiMasker
from nilearn import image

/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-07 14:04:08.962756: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1


In [7]:
subject = 2
bids_folder= '/shares/zne.uzh/mrenke/ds-stressrisk'

In [19]:

# Get paradigm/data/model
sub = Subject(subject, bids_folder=bids_folder)
behavior = sub.get_behavior(sessions=None, drop_no_responses=False).reset_index('session') # session will be range
paradigm = behavior[['n1', 'session']].rename(columns={'n1':'x' }) #,'session':'range'
#if not gaussian:
paradigm['x'] = np.log(paradigm['x']) # as before
paradigm['x'] = paradigm['x'].astype(np.float32)
print(paradigm.describe())

                x     session
count  240.000000  240.000000
mean     3.030885    1.500000
std      0.648520    0.501045
min      1.945910    1.000000
25%      2.639057    1.000000
50%      2.995732    1.500000
75%      3.417332    2.000000
max      4.430817    2.000000


/home/mrenke/git/stress_risk/stress_risk/utils/data.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df.choice.isnull(), 'chose_risky'] = np.nan


In [13]:
roi='NPC_R'
source_key_vselect = 'encoding_model.cv.denoise.retroicor'
subject = f'{int(subject):02d}'

# get average cv-r2 map from seesion 1 for voxel selection
session1 = 1
ips_mask = sub.get_volume_mask(roi=roi, session=1, epi_space=True) # anat from session1
ips_masker = NiftiMasker(mask_img=ips_mask)
im_cvr2_fn = op.join(bids_folder, 'derivatives', source_key_vselect, f'sub-{subject}', f'ses-{session1}','func', f'sub-{subject}_ses-{session1}_desc-cvr2.optim_space-T1w_pars.nii.gz')
im_cvr2 = image.load_img(im_cvr2_fn)
cv_r2 = pd.DataFrame(ips_masker.fit_transform(im_cvr2))
r2_mask = cv_r2 > 0.0
r2_mask = r2_mask.to_numpy().T
masker = NiftiMasker(mask_img=r2_mask)
n_voxels = r2_mask.sum()
print(f'Number of voxels: {n_voxels}')

/data/mrenke/conda/envs/numrefields/lib/python3.9/site-packages/nilearn/image/resampling.py:627: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(img, resampled_data, target_affine)


Number of voxels: 53


In [15]:
# single trial functional brain data
source_key_glm = 'glm_stim1.denoise.retroicor'

data_s1 = op.join(bids_folder, 'derivatives', source_key_glm,
                f'sub-{subject}', f'ses-1', 'func', f'sub-{subject}_ses-1_task-risk_space-T1w_desc-stims1_pe.nii.gz')
data_s2 = op.join(bids_folder, 'derivatives', source_key_glm,
                f'sub-{subject}', f'ses-2', 'func', f'sub-{subject}_ses-2_task-risk_space-T1w_desc-stims1_pe.nii.gz')
data = np.concatenate([ips_masker.fit_transform(data_s1), ips_masker.fit_transform(data_s2)], axis=0)


In [16]:
data.shape

(240, 1072)

In [20]:
paradigm

x  session
subject run trial_nr                   
2       1   1         2.890372        1
            2         3.135494        1
            3         3.713572        1
            4         3.970292        1
            5         3.044523        1
...                        ...      ...
        6   116       2.639057        2
            117       3.332205        2
            118       2.995732        2
            119       2.995732        2
            120       3.332205        2

[240 rows x 2 columns]

In [21]:
data = pd.DataFrame(data, index=paradigm.index)

# now check which subjects have to be rerun


In [ ]:

import glob
import re
import pandas as pd
from os import listdir

bids_folder = '/shares/zne.uzh/mrenke/ds-stressrisk'
subject_list = [int(f[4:]) for f in listdir(bids_folder) if f[0:3] == 'sub' and len(f) == 6]
all_subjects_set = set(subject_list)


In [ ]:
key = 'encoding_model.model6.retroicor'
fns = glob.glob(f'{bids_folder}/derivatives/{key}/sub-*/func/*_desc-r2.optim_space-T1w_pars.npy')
# Correct the regex pattern
reg = re.compile(r'.*/sub-(?P<subject>[0-9]+)_desc-r2\.optim_space-T1w_pars\.npy')

data = []
for fn in fns:
    match = reg.match(fn)
    if match:  # Ensure the regex matches before using groupdict()
        entry = match.groupdict()
        entry['fn'] = fn
        data.append(entry)


data = pd.DataFrame(data)

# Convert subject to int for sorting
data['subject'] = data['subject'].astype(int)

# Set index and print overview
df = data.set_index('subject')
df

In [29]:
df_subjects_set = set(df.index)
missing_subjects = all_subjects_set - df_subjects_set
missing_subjects =list(missing_subjects)
print(*missing_subjects, sep=',')

3,5,12,13,19,23,26,28,30,34,35,36,37,40,43,44,45,46,47,50,55,57,61
